In [6]:
import os
import numpy as np
import librosa
import openai
import torch
from google.cloud import speech
from scipy.io.wavfile import write
import pyaudio
import wave

# 환경 변수 설정 (Google Cloud 인증 파일 경로)
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "path_to_your_google_cloud_credentials.json"
openai.api_key = 'your_openai_api_key'

# 모델 로드 함수
def load_model(model_name, map_location='cpu'):
    if model_name == 'tacotron2':
        tacotron2 = torch.hub.load('NVIDIA/DeepLearningExamples:torchhub', 'nvidia_tacotron2', map_location=map_location)
        checkpoint = torch.hub.get_dir() + '/checkpoints/nvidia_tacotron2pyt_fp32_20190427'
        state_dict = torch.load(checkpoint, map_location=map_location)['state_dict']
        tacotron2.load_state_dict(state_dict)
        return tacotron2
    elif model_name == 'waveglow':
        waveglow = torch.hub.load('NVIDIA/DeepLearningExamples:torchhub', 'nvidia_waveglow', map_location=map_location)
        checkpoint = torch.hub.get_dir() + '/checkpoints/nvidia_waveglowpyt_fp32_20190427'
        state_dict = torch.load(checkpoint, map_location=map_location)['state_dict']
        waveglow.load_state_dict(state_dict)
        return waveglow

# Tacotron2 및 WaveGlow 모델 로드 (CPU로 로드)
tacotron2 = load_model('tacotron2', map_location='cpu')
waveglow = load_model('waveglow', map_location='cpu')
tacotron2.to('cpu').eval()
waveglow.to('cpu').eval()

# 음성 인식 함수
def transcribe_speech(audio_file_path):
    client = speech.SpeechClient()

    with open(audio_file_path, 'rb') as audio_file:
        content = audio_file.read()

    audio = speech.RecognitionAudio(content=content)
    config = speech.RecognitionConfig(
        encoding=speech.RecognitionConfig.AudioEncoding.LINEAR16,
        sample_rate_hertz=16000,
        language_code="ko-KR"  # 한국어 설정
    )

    response = client.recognize(config=config, audio=audio)

    for result in response.results:
        return result.alternatives[0].transcript

# 음성 크기 분석 함수
def analyze_volume(audio_file_path):
    audio, sr = librosa.load(audio_file_path)
    volume = librosa.amplitude_to_db(abs(audio))
    avg_volume = np.mean(volume)
    return avg_volume

# 텍스트 생성 함수
def generate_response(prompt):
    response = openai.Completion.create(
        engine="text-davinci-003",
        prompt=prompt,
        max_tokens=150
    )
    return response.choices[0].text.strip()

# 음성 합성 함수
def synthesize_speech(text, volume_scale=1.0):
    # 텍스트를 시퀀스로 변환
    sequence = torch.tensor(tacotron2.text_to_sequence(text, ['english_cleaners']))[None, :].to('cpu')

    # Tacotron2로 멜 스펙트로그램 생성
    with torch.no_grad():
        mel_outputs, mel_outputs_postnet, _, alignments = tacotron2.infer(sequence)

    # WaveGlow로 오디오 생성
    with torch.no_grad():
        audio = waveglow.infer(mel_outputs_postnet, sigma=0.666).cpu()

    # 볼륨 조정
    audio = audio * volume_scale

    # 오디오 저장
    audio_numpy = audio[0].data.cpu().numpy()
    rate = 22050  # Sample rate
    write("output.wav", rate, audio_numpy)

# 마이크로 음성 입력 받기
def record_audio(output_filename, record_seconds=5):
    chunk = 1024  # Record in chunks of 1024 samples
    sample_format = pyaudio.paInt16  # 16 bits per sample
    channels = 1
    fs = 16000  # Record at 16000 samples per second

    p = pyaudio.PyAudio()  # Create an interface to PortAudio

    print('Recording')

    stream = p.open(format=sample_format,
                    channels=channels,
                    rate=fs,
                    frames_per_buffer=chunk,
                    input=True)

    frames = []  # Initialize array to store frames

    # Store data in chunks for record_seconds
    for i in range(0, int(fs / chunk * record_seconds)):
        data = stream.read(chunk)
        frames.append(data)

    # Stop and close the stream
    stream.stop_stream()
    stream.close()
    # Terminate the PortAudio interface
    p.terminate()

    print('Finished recording')

    # Save the recorded data as a WAV file
    wf = wave.open(output_filename, 'wb')
    wf.setnchannels(channels)
    wf.setsampwidth(p.get_sample_size(sample_format))
    wf.setframerate(fs)
    wf.writeframes(b''.join(frames))
    wf.close()

# 마이크로부터 음성을 받아 처리하는 함수
def process_audio():
    input_audio_path = "input.wav"

    # 1. 마이크로부터 음성 입력 받기
    record_audio(input_audio_path)

    # 2. 음성 인식
    transcribed_text = transcribe_speech(input_audio_path)
    print("Transcribed Text:", transcribed_text)

    # 3. 음성 크기 분석
    volume = analyze_volume(input_audio_path)
    print("Average Volume (dB):", volume)

    # 볼륨 스케일 조정 (예: -60dB ~ 0dB 범위)
    volume_scale = (volume + 60) / 60
    volume_scale = max(0.1, min(2.0, volume_scale))  # 0.1 ~ 2.0 범위로 클램프

    # 4. 텍스트 생성
    generated_text = generate_response(transcribed_text)
    print("Generated Text:", generated_text)

    # 5. 텍스트 음성 변환
    synthesize_speech(generated_text, volume_scale)
    print("Generated speech saved to output.wav")

    # 6. 생성된 음성 재생
    play_audio("output.wav")

# 음성 재생 함수
def play_audio(file_path):
    chunk = 1024
    wf = wave.open(file_path, 'rb')
    p = pyaudio.PyAudio()

    stream = p.open(format=p.get_format_from_width(wf.getsampwidth()),
                    channels=wf.getnchannels(),
                    rate=wf.getframerate(),
                    output=True)

    data = wf.readframes(chunk)

    while data:
        stream.write(data)
        data = wf.readframes(chunk)

    stream.stop_stream()
    stream.close()
    p.terminate()

# 테스트 실행
process_audio()


Using cache found in /Users/kimsoogi/.cache/torch/hub/NVIDIA_DeepLearningExamples_torchhub


RuntimeError: Attempting to deserialize object on a CUDA device but torch.cuda.is_available() is False. If you are running on a CPU-only machine, please use torch.load with map_location=torch.device('cpu') to map your storages to the CPU.